# Botswana Hybrid Spatial Split Training

Mirrors `09_salinas_spatial_split_training.ipynb` with the dataset changed to `botswana`.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from dfm.data.hyperspectral_benchmarks import (
    BOTSWANA_CLASS_NAMES,
    HyperspectralPatchDataset,
    labeled_pixel_indices,
    load_hyperspectral_benchmark,
)

from dfm.models.hybrid import HybridSpatialSpectralClassifier
from dfm.training.metrics import accuracy_score, macro_f1_score
from dfm.training.profiling import count_parameters


In [ ]:
DATASET_ID = "botswana"
CLASS_NAMES = BOTSWANA_CLASS_NAMES
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

# Put .mat files in data/raw/botswana or data/botswana.
data_dir = PROJECT_ROOT / "data"
scene = load_hyperspectral_benchmark(DATASET_ID, data_dir)
rows, cols, labels = labeled_pixel_indices(scene.labels)

print("Dataset:", scene.name)
print("Cube shape (H, W, C):", scene.cube.shape)
print("Label map shape:", scene.labels.shape)
print("Bands:", scene.bands)
print("Classes:", len(scene.class_names))


KeyError: 'Botswana'

In [ ]:
def spatial_train_val_test_split(
    rows,
    cols,
    labels,
    block_size=32,
    train_fraction=0.6,
    val_fraction=0.2,
    seed=42,
):
    """
    Split labeled pixels into spatially separated train/validation/test sets.

    The split is performed at the spatial-block level, not independently at the pixel level.
    """

    if train_fraction + val_fraction >= 1.0:
        raise ValueError("train_fraction + val_fraction must be < 1.")

    block_rows = rows // block_size
    block_cols = cols // block_size
    block_ids = np.stack([block_rows, block_cols], axis=1)
    unique_blocks = np.unique(block_ids, axis=0)

    rng = np.random.default_rng(seed)
    rng.shuffle(unique_blocks)

    n_blocks = len(unique_blocks)
    n_train = int(round(train_fraction * n_blocks))
    n_val = int(round(val_fraction * n_blocks))

    train_blocks = unique_blocks[:n_train]
    val_blocks = unique_blocks[n_train:n_train + n_val]
    test_blocks = unique_blocks[n_train + n_val:]

    def get_indices(blocks):
        mask = np.zeros(len(rows), dtype=bool)
        for br, bc in blocks:
            mask |= (block_rows == br) & (block_cols == bc)
        return np.where(mask)[0]

    train_indices = get_indices(train_blocks)
    val_indices = get_indices(val_blocks)
    test_indices = get_indices(test_blocks)

    rng.shuffle(train_indices)
    rng.shuffle(val_indices)
    rng.shuffle(test_indices)

    return train_indices, val_indices, test_indices


In [ ]:
def find_spatial_split(
    rows,
    cols,
    labels,
    block_size=32,
    train_fraction=0.60,
    val_fraction=0.20,
    seed=42,
    n_trials=10000,
):
    rng = np.random.default_rng(seed)

    block_rows = rows // block_size
    block_cols = cols // block_size
    block_keys = np.stack([block_rows, block_cols], axis=1)
    unique_blocks = np.unique(block_keys, axis=0)

    n_blocks = len(unique_blocks)
    n_train = round(train_fraction * n_blocks)
    n_val = round(val_fraction * n_blocks)

    best_score = np.inf
    best_assignment = None
    best_missing_classes = None

    block_counts = []
    for br, bc in unique_blocks:
        mask = (block_rows == br) & (block_cols == bc)
        counts = np.bincount(labels[mask], minlength=len(CLASS_NAMES))
        block_counts.append(counts)

    block_counts = np.asarray(block_counts)
    total_class_counts = block_counts.sum(axis=0)
    present_classes = total_class_counts > 0
    overall_dist = total_class_counts / total_class_counts.sum()

    for _ in range(n_trials):
        permutation = rng.permutation(n_blocks)

        train_ids = permutation[:n_train]
        val_ids = permutation[n_train:n_train + n_val]
        test_ids = permutation[n_train + n_val:]

        train_counts = block_counts[train_ids].sum(axis=0)
        val_counts = block_counts[val_ids].sum(axis=0)
        test_counts = block_counts[test_ids].sum(axis=0)

        train_missing = present_classes & (train_counts == 0)
        val_missing = present_classes & (val_counts == 0)
        test_missing = present_classes & (test_counts == 0)
        missing_classes = (
            train_missing.sum()
            + val_missing.sum()
            + test_missing.sum()
        )

        train_dist = train_counts / max(train_counts.sum(), 1)
        val_dist = val_counts / max(val_counts.sum(), 1)
        test_dist = test_counts / max(test_counts.sum(), 1)

        distribution_error = (
            np.mean(np.abs(train_dist - overall_dist))
            + np.mean(np.abs(val_dist - overall_dist))
            + np.mean(np.abs(test_dist - overall_dist))
        )

        total_pixels = len(labels)
        target_train = train_fraction * total_pixels
        target_val = val_fraction * total_pixels
        target_test = (1 - train_fraction - val_fraction) * total_pixels

        size_error = (
            abs(train_counts.sum() - target_train) / total_pixels
            + abs(val_counts.sum() - target_val) / total_pixels
            + abs(test_counts.sum() - target_test) / total_pixels
        )

        # Prefer the original strict behavior when possible. If it is impossible
        # for sparse/small-class datasets, use the most class-balanced spatial split.
        missing_penalty = missing_classes * 10.0
        score = distribution_error + size_error + missing_penalty

        if score < best_score:
            best_score = score
            best_missing_classes = int(missing_classes)
            best_assignment = (
                unique_blocks[train_ids].copy(),
                unique_blocks[val_ids].copy(),
                unique_blocks[test_ids].copy(),
            )

            if missing_classes == 0:
                # This is the same validity condition used by the Salinas notebook.
                # Keep searching, but now only lower distribution/size error can win.
                pass

    if best_assignment is None:
        raise RuntimeError(
            "Could not find a spatial split. Try reducing block_size or checking labels."
        )

    train_blocks, val_blocks, test_blocks = best_assignment

    def blocks_to_indices(blocks):
        mask = np.zeros(len(rows), dtype=bool)
        for br, bc in blocks:
            mask |= (block_rows == br) & (block_cols == bc)
        return np.where(mask)[0]

    train_indices = blocks_to_indices(train_blocks)
    val_indices = blocks_to_indices(val_blocks)
    test_indices = blocks_to_indices(test_blocks)

    rng.shuffle(train_indices)
    rng.shuffle(val_indices)
    rng.shuffle(test_indices)

    if best_missing_classes:
        print(
            "Warning: no perfect class-aware spatial split was found; "
            f"using best split with {best_missing_classes} missing class/split placements."
        )

    return (
        train_indices,
        val_indices,
        test_indices,
        train_blocks,
        val_blocks,
        test_blocks,
        best_score,
    )


In [ ]:
from collections import Counter


def print_class_distribution(indices, name):
    class_labels = labels[indices]
    counts = Counter(class_labels)

    print(f"\n{name}")
    print("-" * 30)

    for class_id in range(len(CLASS_NAMES)):
        print(
            f"{class_id + 1:2d} "
            f"{CLASS_NAMES[class_id]:30s} "
            f"{counts.get(class_id, 0)}"
        )


In [ ]:
(
    train_indices,
    val_indices,
    test_indices,
    train_blocks,
    val_blocks,
    test_blocks,
    split_score,
) = find_spatial_split(
    rows,
    cols,
    labels,
    block_size=32,
    train_fraction=0.60,
    val_fraction=0.20,
    seed=SEED,
    n_trials=10000,
)

print("Split score:", split_score)
print("Train samples:", len(train_indices))
print("Validation samples:", len(val_indices))
print("Test samples:", len(test_indices))

print_class_distribution(train_indices, "TRAIN")
print_class_distribution(val_indices, "VALIDATION")
print_class_distribution(test_indices, "TEST")


In [ ]:
split_map = np.zeros_like(scene.labels)

for idx in train_indices:
    split_map[rows[idx], cols[idx]] = 1
for idx in val_indices:
    split_map[rows[idx], cols[idx]] = 2
for idx in test_indices:
    split_map[rows[idx], cols[idx]] = 3

plt.figure(figsize=(12, 8))
plt.imshow(split_map)
plt.title(f"{scene.name} Class-Aware Spatial Train / Validation / Test Split")
plt.colorbar(ticks=[0, 1, 2, 3], label="0=Background, 1=Train, 2=Validation, 3=Test")
plt.show()


In [ ]:
patch_size = 15

train_dataset = HyperspectralPatchDataset(scene, indices=train_indices, patch_size=patch_size)
val_dataset = HyperspectralPatchDataset(scene, indices=val_indices, patch_size=patch_size)
test_dataset = HyperspectralPatchDataset(scene, indices=test_indices, patch_size=patch_size)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=0)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
model = HybridSpatialSpectralClassifier(
    in_channels=scene.bands,
    num_classes=len(CLASS_NAMES),
    feature_dim=128,
    spectral_depth=2,
    spectral_heads=4,
    fusion="gated",
    dropout=0.1,
    max_bands=256,
).to(device)

print(model)
print("Trainable parameters:", count_parameters(model))


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)


In [ ]:
def train_one_epoch(model, loader):
    model.train()

    total_loss = 0.0
    total = 0

    for x, y in tqdm(loader, leave=False):
        x = x.to(device=device, dtype=torch.float32)
        y = y.to(device=device, dtype=torch.long)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.shape[0]
        total += x.shape[0]

    return total_loss / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    all_true = []
    all_pred = []

    for x, y in tqdm(loader, leave=False):
        x = x.to(device=device, dtype=torch.float32)
        logits = model(x)
        pred = logits.argmax(dim=1)

        all_pred.append(pred.cpu().numpy())
        all_true.append(y.numpy())

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": macro_f1_score(y_true, y_pred, num_classes=len(CLASS_NAMES)),
    }


In [ ]:
model.eval()
x, y = next(iter(val_loader))
x = x.to(device=device, dtype=torch.float32)

with torch.no_grad():
    logits = model(x)

print("Input:", x.shape)
print("Output:", logits.shape)


In [ ]:
epochs = 50
patience = 50

history = []
best_macro_f1 = -1.0
best_epoch = None
best_state = None
epochs_without_improvement = 0

for epoch in range(1, epochs + 1):
    train_loss = train_one_epoch(model, train_loader)
    val_metrics = evaluate(model, val_loader)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"Loss: {train_loss:.4f} | "
        f"Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val Macro-F1: {val_metrics['macro_f1']:.4f}"
    )

    if val_metrics["macro_f1"] > best_macro_f1:
        best_macro_f1 = val_metrics["macro_f1"]
        best_epoch = epoch
        epochs_without_improvement = 0
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }
        print("  New best model")
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print(f"\nEarly stopping at epoch {epoch}")
        break

history_df = pd.DataFrame(history)

print("\nBest epoch:", best_epoch)
print("Best validation Macro-F1:", best_macro_f1)
history_df


In [ ]:
model.load_state_dict(best_state)
print(f"Restored best model from epoch {best_epoch}")

test_metrics = evaluate(model, test_loader)

print("\n" + "=" * 50)
print("FINAL SPATIAL TEST RESULTS")
print("=" * 50)
print(f"Accuracy : {test_metrics['accuracy']:.4f}")
print(f"Macro-F1 : {test_metrics['macro_f1']:.4f}")


In [ ]:
outputs_dir = PROJECT_ROOT / "outputs" / DATASET_ID
outputs_dir.mkdir(parents=True, exist_ok=True)

history_df.to_csv(outputs_dir / "hybrid_spatial_history.csv", index=False)

checkpoint_path = outputs_dir / "hybrid_spatial_best.pt"

if best_state is None:
    raise RuntimeError("best_state is None; the best model weights are not available in memory.")

torch.save(
    {
        "model_state_dict": best_state,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_macro_f1,
        "test_accuracy": test_metrics["accuracy"],
        "test_macro_f1": test_metrics["macro_f1"],
        "class_names": CLASS_NAMES,
        "patch_size": patch_size,
        "bands": scene.bands,
        "seed": SEED,
        "dataset": DATASET_ID,
    },
    checkpoint_path,
)

np.savez(
    outputs_dir / f"{DATASET_ID}_spatial_split_seed42.npz",
    train_indices=train_indices,
    val_indices=val_indices,
    test_indices=test_indices,
)

print("Saved:", checkpoint_path)


In [ ]:
checkpoint_check = torch.load(checkpoint_path, map_location="cpu")

print("Best epoch:", checkpoint_check["best_epoch"])
print("Best Val Macro-F1:", checkpoint_check["best_val_macro_f1"])
print("Weights type:", type(checkpoint_check["model_state_dict"]))
print("Number of tensors:", len(checkpoint_check["model_state_dict"]))


In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 5))

ax1.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Train Loss")
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(history_df["epoch"], history_df["val_accuracy"], marker="o", label="Validation Accuracy")
ax2.plot(history_df["epoch"], history_df["val_macro_f1"], marker="o", label="Validation Macro-F1")

if best_epoch is not None:
    ax1.axvline(best_epoch, linestyle="--", alpha=0.35, label="Best Epoch")

ax2.set_ylabel("Validation Score")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right")

plt.title(f"{scene.name} Hybrid - Spatial Split Training")
plt.tight_layout()
plt.show()


In [ ]:
@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()

    all_true = []
    all_pred = []

    for x, y in tqdm(loader, desc="Collecting test predictions"):
        x = x.to(device=device, dtype=torch.float32)
        logits = model(x)
        pred = logits.argmax(dim=1).cpu().numpy()

        all_pred.append(pred)
        all_true.append(y.numpy())

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)

    return y_true, y_pred


y_test, y_pred = collect_predictions(model, test_loader)

np.save(outputs_dir / "hybrid_spatial_y_test.npy", y_test)
np.save(outputs_dir / "hybrid_spatial_y_pred.npy", y_pred)

print("Test samples:", len(y_test))
print("Predictions:", len(y_pred))
print("Predictions saved.")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

report = classification_report(
    y_test,
    y_pred,
    labels=np.arange(len(CLASS_NAMES)),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)

per_class_df = pd.DataFrame(report).T
per_class_df.to_csv(outputs_dir / "hybrid_spatial_per_class_metrics.csv")
per_class_df


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=np.arange(len(CLASS_NAMES)))

cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
cm_df.to_csv(outputs_dir / "hybrid_spatial_confusion_matrix.csv")
cm_df
